In [14]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import numpy as np, pandas as pd
from src.config import PROCESSED
from src.indicators import atr
from src.strategies import señales_reversion
from src.features import crear_features, crear_dataset
import lightgbm as lgb

In [16]:
df = pd.read_parquet(PROCESSED)
df["atr"] = atr(df, 14)

In [17]:
señ = señales_reversion(df, horas=tuple(range(24)), k=1.5)          # regla primaria: todas las horas
feats = crear_features(df)
params = dict(sl_mult=2.0, tp_mult=1.5, max_velas=12, coste=0.25)
ds = crear_dataset(df, señ, feats, **params)

In [18]:
print(ds.shape, ds["y"].mean().round(3), ds["R"].mean().round(3))
ds.groupby(ds["datetime"].dt.year)[["y", "R"]].agg(["mean", "count"]).round(3)

(34086, 17) 0.542 -0.089


y            R      
           mean count   mean count
datetime                          
2020      0.560  4751 -0.063  4751
2021      0.533  4989 -0.124  4989
2022      0.539  5147 -0.117  5147
2023      0.547  5058 -0.114  5058
2024      0.536  5150 -0.104  5150
2025      0.536  5097 -0.063  5097
2026      0.546  3894 -0.023  3894

In [19]:



FEATURES = ["direccion", "z", "z_dia", "pend_dia", "atr_ratio", "ret_1", "ret_3", "ret_12",
            "rango_12", "coste_rel", "hora_sin", "hora_cos", "dia_sem"]
ds["año"] = ds["datetime"].dt.year

def nuevo_modelo():
    return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15,
                              min_child_samples=200, subsample=0.8, subsample_freq=1,
                              colsample_bytree=0.8, reg_lambda=1.0, verbose=-1, random_state=0)

filas, importancias = [], []
for año in range(2022, 2027):
    inicio_test = pd.Timestamp(f"{año}-01-01")
    train = ds[ds["datetime"] < inicio_test - pd.Timedelta(days=2)]          # purga de 2 días
    test = ds[ds["año"] == año]

    modelo = nuevo_modelo().fit(train[FEATURES], train["y"])
    ultimo = train[train["año"] == año - 1]
    umbral = np.quantile(modelo.predict_proba(ultimo[FEATURES])[:, 1], 0.80)  # umbral sin mirar el test

    prob = modelo.predict_proba(test[FEATURES])[:, 1]
    elegidos = test[prob > umbral]

    # Control: filtro aleatorio del mismo tamaño
    rng = np.random.default_rng(0)
    azar = [test["R"].sample(len(elegidos), random_state=int(s)).mean()
            for s in rng.integers(0, 1e9, 200)]
    p = (np.array(azar) >= elegidos["R"].mean()).mean()

    filas.append({"año": año, "candidatas": len(test), "elegidas": len(elegidos),
                  "R_todas": test["R"].mean(), "R_ML": elegidos["R"].mean(),
                  "R_azar": np.mean(azar), "ventaja": elegidos["R"].mean() - np.mean(azar),
                  "p": p})
    importancias.append(pd.Series(modelo.feature_importances_, index=FEATURES, name=año))

res_ml = pd.DataFrame(filas).set_index("año")
res_ml.round(3)

,candidatas,elegidas,R_todas,R_ML,R_azar,ventaja,p
año,,,,,,,
2022,5147,1020,-0.117,-0.098,-0.118,0.020,0.21
2023,5058,858,-0.114,-0.087,-0.111,0.023,0.20
2024,5150,1178,-0.104,-0.091,-0.104,0.014,0.24
2025,5097,1348,-0.063,-0.057,-0.063,0.006,0.38
2026,3894,1091,-0.023,-0.006,-0.021,0.015,0.28


In [20]:
pd.concat(importancias, axis=1).mean(axis=1).sort_values(ascending=False)

rango_12     467.0
coste_rel    455.8
atr_ratio    450.0
z            442.0
ret_1        408.4
ret_12       404.8
ret_3        397.8
z_dia        342.2
pend_dia     296.6
hora_cos     214.2
hora_sin     202.2
dia_sem      117.4
direccion      1.6
dtype: float64

In [21]:
# "a favor del trade": positivo = el precio se mueve en la dirección del trade
for col in ["z", "z_dia", "pend_dia", "ret_1", "ret_3", "ret_12"]:
    ds[f"{col}_dir"] = ds[col] * ds["direccion"]

FEATURES_V2 = ["z_dir", "z_dia_dir", "pend_dia_dir", "ret_1_dir", "ret_3_dir", "ret_12_dir",
               "atr_ratio", "rango_12", "coste_rel", "hora_sin", "hora_cos", "dia_sem"]

In [22]:
def nuevo_regresor():
    return lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=15,
                             min_child_samples=200, subsample=0.8, subsample_freq=1,
                             colsample_bytree=0.8, reg_lambda=1.0, verbose=-1, random_state=0)

filas, importancias, elegidos_todos = [], [], []
for año in range(2022, 2027):
    inicio_test = pd.Timestamp(f"{año}-01-01")
    train = ds[ds["datetime"] < inicio_test - pd.Timedelta(days=2)]
    test = ds[ds["año"] == año]

    modelo = nuevo_regresor().fit(train[FEATURES_V2], train["R"])           # predice la R esperada
    ultimo = train[train["año"] == año - 1]
    umbral = np.quantile(modelo.predict(ultimo[FEATURES_V2]), 0.95)         # top 5 %, sin mirar el test

    pred = modelo.predict(test[FEATURES_V2])
    elegidos = test[pred > umbral]
    elegidos_todos.append(elegidos)

    rng = np.random.default_rng(0)
    azar = [test["R"].sample(len(elegidos), random_state=int(s)).mean()
            for s in rng.integers(0, 1e9, 200)]
    filas.append({"año": año, "elegidas": len(elegidos),
                  "R_todas": test["R"].mean(), "R_ML": elegidos["R"].mean(),
                  "R_azar": np.mean(azar), "ventaja": elegidos["R"].mean() - np.mean(azar),
                  "p": (np.array(azar) >= elegidos["R"].mean()).mean()})
    importancias.append(pd.Series(modelo.booster_.feature_importance("gain"),
                                  index=FEATURES_V2, name=año))

res_v2 = pd.DataFrame(filas).set_index("año")
res_v2.round(3)

,elegidas,R_todas,R_ML,R_azar,ventaja,p
año,,,,,,
2022,307,-0.117,0.031,-0.116,0.147,0.000
2023,153,-0.114,-0.103,-0.113,0.009,0.440
2024,499,-0.104,-0.108,-0.102,-0.006,0.560
2025,586,-0.063,-0.055,-0.064,0.008,0.395
2026,469,-0.023,-0.028,-0.022,-0.005,0.575


In [23]:
el = pd.concat(elegidos_todos)
r = el["R"]
print(f"Conjunto OOS 2022–26: n={len(r)}  R_ML={r.mean():.3f}  "
      f"t={r.mean() / r.std() * np.sqrt(len(r)):.2f}  R_todas={ds[ds['año'] >= 2022]['R'].mean():.3f}")
pd.concat(importancias, axis=1).mean(axis=1).sort_values(ascending=False).round(0)

Conjunto OOS 2022–26: n=2014  R_ML=-0.052  t=-3.04  R_todas=-0.087


coste_rel       2118.0
z_dir           1491.0
rango_12        1491.0
ret_3_dir       1479.0
atr_ratio       1479.0
ret_12_dir      1388.0
ret_1_dir       1225.0
z_dia_dir       1083.0
pend_dia_dir     972.0
hora_cos         761.0
hora_sin         681.0
dia_sem          392.0
dtype: float64